In [ ]:
# -*- coding: utf-8 -*-
"""03_Text_Processing.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1tDQUbf2yd7WJln6W1pCwy01cefn88Hl3
"""

import pandas as pd
import sys
from google.colab import drive
import re
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, save_npz

# --- 1. Setup Environment and Install Library ---
print("Setting up the environment...")
drive.mount('/content/drive', force_remount=True)
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer

BASE_PATH = '/content/drive/MyDrive/AMLC_2025/'
DATA_PATH = BASE_PATH + 'data/'

print("\nLoading full datasets...")
train_df = pd.read_csv(DATA_PATH + 'train.csv')
test_df = pd.read_csv(DATA_PATH + 'test.csv')
print("Setup Complete.")

# --- 2. Text Parsing and Cleaning ---
print("\nParsing and cleaning the 'catalog_content' column...")
def parse_catalog_content_v2(text):
    text = str(text) + "\n"
    item_name_match = re.search(r"Item Name: (.*?)\n", text, re.DOTALL)
    item_name = item_name_match.group(1).strip() if item_name_match else ""
    value_match = re.search(r"Value: (.*?)\n", text, re.DOTALL)
    try: value = float(value_match.group(1).strip()) if value_match else np.nan
    except (ValueError, TypeError): value = np.nan
    unit_match = re.search(r"Unit: (.*?)\n", text, re.DOTALL)
    unit = unit_match.group(1).strip() if unit_match else "unknown"
    bullet_points = " ".join(re.findall(r"Bullet Point.*?: (.*?)\n", text, re.DOTALL))
    prod_desc_match = re.search(r"Product Description: (.*?)\nValue:", text, re.DOTALL)
    if not prod_desc_match: prod_desc_match = re.search(r"Product Description: (.*)", text, re.DOTALL)
    prod_desc = prod_desc_match.group(1).strip() if prod_desc_match else ""
    description = f"{bullet_points} {prod_desc}".strip()
    return item_name, value, unit, description

train_df[['item_name', 'value', 'unit', 'description']] = train_df['catalog_content'].apply(lambda x: pd.Series(parse_catalog_content_v2(x)))
test_df[['item_name', 'value', 'unit', 'description']] = test_df['catalog_content'].apply(lambda x: pd.Series(parse_catalog_content_v2(x)))
train_df.drop(columns=['catalog_content'], inplace=True)
test_df.drop(columns=['catalog_content'], inplace=True)
median_value = train_df['value'].median()
train_df['value'].fillna(median_value, inplace=True)
test_df['value'].fillna(median_value, inplace=True)
train_df['unit'].fillna('unknown', inplace=True)
test_df['unit'].fillna('unknown', inplace=True)
print("Text parsing and cleaning complete.")

# --- 3. Generate Sentence Transformer Embeddings ---
print("\nLoading Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

# Combine the item_name and description for a richer input
train_sentences = (train_df['item_name'] + ". " + train_df['description']).fillna('').tolist()
test_sentences = (test_df['item_name'] + ". " + test_df['description']).fillna('').tolist()

print("\nGenerating text embeddings (this will take 10-20 minutes)...")
X_train_text_embeddings = model.encode(train_sentences, show_progress_bar=True)
X_test_text_embeddings = model.encode(test_sentences, show_progress_bar=True)
print("Text embedding generation complete.")

# --- 4. Create Other Features and Save Everything ---
print("\nCreating final features and saving files...")
encoder = OneHotEncoder(handle_unknown='ignore')
X_train_unit = encoder.fit_transform(train_df[['unit']])
X_test_unit = encoder.transform(test_df[['unit']])

# Combine the dense sentence embeddings, the numerical value, and the sparse unit features
X_train_text_features = hstack([X_train_text_embeddings, train_df[['value']], X_train_unit])
X_test_text_features = hstack([X_test_text_embeddings, test_df[['value']], X_test_unit])

# Save the final feature files
save_npz(DATA_PATH + 'X_train_text_features_bert.npz', X_train_text_features)
save_npz(DATA_PATH + 'X_test_text_features_bert.npz', X_test_text_features)

# Save the target variable (price) and test IDs for the final modeling notebook
y_train_full = train_df['price'].values
np.save(DATA_PATH + 'y_train_full.npy', y_train_full)
test_ids = test_df[['sample_id']]
test_ids.to_csv(DATA_PATH + 'test_ids.csv', index=False)
print("Text feature files saved successfully.")

print("\n✅ Notebook 2 ('02_Text_Processing.ipynb') is COMPLETE.")
print(f"Final text feature shape: {X_train_text_features.shape}")

